# Notebook 1 - Multi-Agent Fake News Pipeline

End-to-end demo of the CrewAI 4-agent sequential pipeline:

`Claim Extractor -> Fact Checker (Wikipedia) -> Bias Detector -> Judge`

Make sure Ollama is running (`ollama serve`) and that the model defined in `.env` is pulled.

In [11]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.config import settings
from src.crew import run_pipeline
from src.dataset import sample_dataset, label_to_str

print('Backend:', settings.backend)
print('Model  :', settings.ollama_model if settings.backend == 'ollama' else settings.openai_model)

Backend: ollama
Model  : qwen2.5:3b


## 1. Hand-picked example

Run the pipeline on a short, clearly fake-sounding article to verify the full flow.

In [12]:
title = 'Scientists confirm the Moon is made of cheese'
body = (
    'In a shocking announcement today, a team of scientists at Harvard University '
    'declared that the Moon is entirely made of aged cheddar cheese. The study, '
    'published in the Journal of Dairy Astronomy, claims that Apollo 11 astronauts '
    'secretly brought back cheese samples in 1969 that have been hidden from the '
    'public for over five decades.'
)

result = run_pipeline(title, body)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9789e471-e505-4516-b227-936131f6f159                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Read the news article below and extract up to 3 concise, VERIFIABLE factual claims. Focus on             │
│  statistics, named entities, concrete events, and dates. Ignore opinions and rhetorical language.               │
│                                                                                                                 │
│  TITLE: Scientists confirm the Moon is made of cheese                                                           │
│                                                                                                                 │
│  BODY:                                                                                                          │
│  In a shocking announcement today, a team of scientists at Harvard University declared that the Moon is         │
│  entirely made of aged cheddar cheese. The study, published in the Journal of Dairy Astronomy, claims that      │
│  Apollo 11 astronauts secretly brought back cheese samples in 1969 that have been hidden from the public for    │
│  over five decades.                                                                                             │
│                                                                                                                 │
│  Return a JSON object matching the ClaimsOutput schema with exactly 3 items in 'claims' (or fewer if the        │
│  article has fewer verifiable claims).                                                                          │
│  ID: 0017ed33-7f6a-4abe-8776-4ded19545ca4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Claim Extractor                                                                                         │
│                                                                                                                 │
│  Task: Read the news article below and extract up to 3 concise, VERIFIABLE factual claims. Focus on             │
│  statistics, named entities, concrete events, and dates. Ignore opinions and rhetorical language.               │
│                                                                                                                 │
│  TITLE: Scientists confirm the Moon is made of cheese                                                           │
│                                                                                                                 │
│  BODY:                                                                                                          │
│  In a shocking announcement today, a team of scientists at Harvard University declared that the Moon is         │
│  entirely made of aged cheddar cheese. The study, published in the Journal of Dairy Astronomy, claims that      │
│  Apollo 11 astronauts secretly brought back cheese samples in 1969 that have been hidden from the public for    │
│  over five decades.                                                                                             │
│                                                                                                                 │
│  Return a JSON object matching the ClaimsOutput schema with exactly 3 items in 'claims' (or fewer if the        │
│  article has fewer verifiable claims).                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Claim Extractor                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  claims=['A team of scientists at Harvard University announced that the Moon is made entirely of aged cheddar   │
│  cheese.', 'The study was published in the Journal of Dairy Astronomy.', 'Apollo 11 astronauts brought back     │
│  cheese samples to Earth in 1969, which have been hidden from public view for over five decades.']              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Read the news article below and extract up to 3 concise, VERIFIABLE factual claims. Focus on             │
│  statistics, named entities, concrete events, and dates. Ignore opinions and rhetorical language.               │
│                                                                                                                 │
│  TITLE: Scientists confirm the Moon is made of cheese                                                           │
│                                                                                                                 │
│  BODY:                                                                                                          │
│  In a shocking announcement today, a team of scientists at Harvard University declared that the Moon is         │
│  entirely made of aged cheddar cheese. The study, published in the Journal of Dairy Astronomy, claims that      │
│  Apollo 11 astronauts secretly brought back cheese samples in 1969 that have been hidden from the public for    │
│  over five decades.                                                                                             │
│                                                                                                                 │
│  Return a JSON object matching the ClaimsOutput schema with exactly 3 items in 'claims' (or fewer if the        │
│  article has fewer verifiable claims).                                                                          │
│  Agent: Claim Extractor                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Judge                                                                                                   │
│                                                                                                                 │
│  Task: You are the Judge. Using the outputs of the previous three agents (claim extraction, fact-checking       │
│  verdicts, and bias analysis), produce a FINAL classification of the article as REAL or FAKE.                   │
│                                                                                                                 │
│  Decision heuristic:                                                                                            │
│    - If most claims are CONTRADICTED -> likely FAKE.                                                            │
│    - If most claims are SUPPORTED and bias_score is low -> likely REAL.                                         │
│    - High bias_score alone is a weak signal (REAL articles can be biased).                                      │
│    - UNVERIFIABLE claims are neutral; weigh them less.                                                          │
│                                                                                                                 │
│  Return a JSON object with: label ('REAL' or 'FAKE'), confidence (0-1), summary (2-3 sentence human-readable    │
│  explanation), and agent_inputs_summary {claim_verdicts: [...], bias_score: float, tone: str}.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Judge                                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  label='FAKE' confidence=0.82 summary='The article contains three clearly contradicted claims, with a high      │
│  bias score indicating sensationalist writing that may undermine the credibility of the content.'               │
│  agent_inputs_summary=AgentInputsSummary(claim_verdicts=['CONTRADICTED', 'CONTRADICTED', 'CONTRADICTED'],       │
│  bias_score=0.7, tone='sensationalist')                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: You are the Judge. Using the outputs of the previous three agents (claim extraction, fact-checking       │
│  verdicts, and bias analysis), produce a FINAL classification of the article as REAL or FAKE.                   │
│                                                                                                                 │
│  Decision heuristic:                                                                                            │
│    - If most claims are CONTRADICTED -> likely FAKE.                                                            │
│    - If most claims are SUPPORTED and bias_score is low -> likely REAL.                                         │
│    - High bias_score alone is a weak signal (REAL articles can be biased).                                      │
│    - UNVERIFIABLE claims are neutral; weigh them less.                                                          │
│                                                                                                                 │
│  Return a JSON object with: label ('REAL' or 'FAKE'), confidence (0-1), summary (2-3 sentence human-readable    │
│  explanation), and agent_inputs_summary {claim_verdicts: [...], bias_score: float, tone: str}.                  │
│  Agent: Judge                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9789e471-e505-4516-b227-936131f6f159                                                                       │
│  Final Output: {"label":"FAKE","confidence":0.82,"summary":"The article contains three clearly contradicted     │
│  claims, with a high bias score indicating sensationalist writing that may undermine the credibility of the     │
│  content.","agent_inputs_summary":{"claim_verdicts":["CONTRADICTED","CONTRADICTED","CONTRADICTED"],"bias_score  │
│  ":0.7,"tone":"sensationalist"}}                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [13]:
print('--- Claims ---')
print(result.claims.model_dump_json(indent=2) if result.claims else '(none)')
print('\n--- Fact Check ---')
print(result.fact_check.model_dump_json(indent=2) if result.fact_check else '(none)')
print('\n--- Bias ---')
print(result.bias.model_dump_json(indent=2) if result.bias else '(none)')
print('\n--- Judge ---')
print(result.judge.model_dump_json(indent=2) if result.judge else '(none)')

--- Claims ---
{
  "claims": [
    "A team of scientists at Harvard University announced that the Moon is made entirely of aged cheddar cheese.",
    "The study was published in the Journal of Dairy Astronomy.",
    "Apollo 11 astronauts brought back cheese samples to Earth in 1969, which have been hidden from public view for over five decades."
  ]
}

--- Fact Check ---
{
  "results": [
    {
      "claim": "A team of scientists at Harvard University announced that the Moon is made entirely of aged cheddar cheese.",
      "verdict": "CONTRADICTED",
      "confidence": 1.0,
      "evidence": "The Wikipedia search for 'Harvard University Moon cheese' confirms that there has never been any announcement or study by scientists at Harvard University regarding the composition of the Moon being made of aged cheddar cheese."
    },
    {
      "claim": "The study was published in the Journal of Dairy Astronomy.",
      "verdict": "CONTRADICTED",
      "confidence": 1.0,
      "evidence": "The 

## 2. Sampled article from the Kaggle dataset

Run one real article from the Kaggle fake/real dataset. (Requires `scripts/download_dataset.py` to have been run once.)

In [14]:
sample = sample_dataset(n=4, random_state=7)
sample[['article_id', 'title', 'label']]

,article_id,title,label
0,27996,Warner's opposition to Trump court nominee giv...,0
1,21461,WOMAN PULLED OVER FOR 51 MPH IN SCHOOL ZONE: “...,1
2,7377,John Oliver Successfully Turns Trump’s ‘Plan’...,1
3,34523,"Biden, Cypriot President Anastasiades discuss ...",0


In [15]:
rec = sample.iloc[0]
print('True label:', label_to_str(rec['label']))
print('Title:', rec['title'])
print()
result = run_pipeline(rec['title'], rec['text'])
print('\nPredicted:', result.judge.label if result.judge else '(none)',
      '| confidence:', getattr(result.judge, 'confidence', None))

True label: REAL
Title: Warner's opposition to Trump court nominee gives Democrats 41 'no' votes



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 69c3ec51-9acd-4429-bcef-19d29cefc129                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Read the news article below and extract up to 3 concise, VERIFIABLE factual claims. Focus on             │
│  statistics, named entities, concrete events, and dates. Ignore opinions and rhetorical language.               │
│                                                                                                                 │
│  TITLE: Warner's opposition to Trump court nominee gives Democrats 41 'no' votes                                │
│                                                                                                                 │
│  BODY:                                                                                                          │
│  WASHINGTON (Reuters) - Democratic Senator Mark Warner on Monday announced opposition to President Donald       │
│  Trump’s Supreme Court nominee, meaning Democrats potentially have the votes needed to block a U.S. Senate      │
│  confirmation vote to give Neil Gorsuch the lifetime job. Warner’s announcement put the number of Democrats     │
│  opposing Gorsuch at 41, the tally needed to use a procedural hurdle called a filibuster that requires 60       │
│  votes to allow a confirmation vote in the 100-seat Senate. It remained unclear, however, if all 41 senators    │
│  who have voiced opposition would back a filibuster. Republicans control the Senate 52-48. If Democrats amass   │
│  the 41 votes to block a confirmation vote that Senate Republicans have planned for Friday, the Republicans     │
│  would then be expected to try to change the chamber’s long-standing rules and allow confirmation by a simple   │
│  majority, a move backed by Trump that is sometimes called the “nuclear option.”                                │
│                                                                                                                 │
│  Return a JSON object matching the ClaimsOutput schema with exactly 3 items in 'claims' (or fewer if the        │
│  article has fewer verifiable claims).                                                                          │
│  ID: 82d94577-dcc8-4d58-b567-6310bfec46e0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Claim Extractor                                                                                         │
│                                                                                                                 │
│  Task: Read the news article below and extract up to 3 concise, VERIFIABLE factual claims. Focus on             │
│  statistics, named entities, concrete events, and dates. Ignore opinions and rhetorical language.               │
│                                                                                                                 │
│  TITLE: Warner's opposition to Trump court nominee gives Democrats 41 'no' votes                                │
│                                                                                                                 │
│  BODY:                                                                                                          │
│  WASHINGTON (Reuters) - Democratic Senator Mark Warner on Monday announced opposition to President Donald       │
│  Trump’s Supreme Court nominee, meaning Democrats potentially have the votes needed to block a U.S. Senate      │
│  confirmation vote to give Neil Gorsuch the lifetime job. Warner’s announcement put the number of Democrats     │
│  opposing Gorsuch at 41, the tally needed to use a procedural hurdle called a filibuster that requires 60       │
│  votes to allow a confirmation vote in the 100-seat Senate. It remained unclear, however, if all 41 senators    │
│  who have voiced opposition would back a filibuster. Republicans control the Senate 52-48. If Democrats amass   │
│  the 41 votes to block a confirmation vote that Senate Republicans have planned for Friday, the Republicans     │
│  would then be expected to try to change the chamber’s long-standing rules and allow confirmation by a simple   │
│  majority, a move backed by Trump that is sometimes called the “nuclear option.”                                │
│                                                                                                                 │
│  Return a JSON object matching the ClaimsOutput schema with exactly 3 items in 'claims' (or fewer if the        │
│  article has fewer verifiable claims).                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Claim Extractor                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  claims=["Democrats have announced opposition to President Donald Trump's Supreme Court nominee, Neil           │
│  Gorsuch.", 'Warner’s announcement puts the number of Democrats opposing Gorsuch at 41, which is the tally      │
│  needed for a filibuster in the Senate.', 'Republicans currently control the Senate with a majority of 52-48    │
│  seats.']                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Judge                                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  label='FAKE' confidence=0.82 summary='The article contains unsupported claims and is biased, suggesting it     │
│  may be fake.' agent_inputs_summary=AgentInputsSummary(claim_verdicts=['CONTRADICTED', 'SUPPORTED',             │
│  'UNVERIFIABLE'], bias_score=0.7, tone='sensationalist')                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: You are the Judge. Using the outputs of the previous three agents (claim extraction, fact-checking       │
│  verdicts, and bias analysis), produce a FINAL classification of the article as REAL or FAKE.                   │
│                                                                                                                 │
│  Decision heuristic:                                                                                            │
│    - If most claims are CONTRADICTED -> likely FAKE.                                                            │
│    - If most claims are SUPPORTED and bias_score is low -> likely REAL.                                         │
│    - High bias_score alone is a weak signal (REAL articles can be biased).                                      │
│    - UNVERIFIABLE claims are neutral; weigh them less.                                                          │
│                                                                                                                 │
│  Return a JSON object with: label ('REAL' or 'FAKE'), confidence (0-1), summary (2-3 sentence human-readable    │
│  explanation), and agent_inputs_summary {claim_verdicts: [...], bias_score: float, tone: str}.                  │
│  Agent: Judge                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 69c3ec51-9acd-4429-bcef-19d29cefc129                                                                       │
│  Final Output: {"label":"FAKE","confidence":0.82,"summary":"The article contains unsupported claims and is      │
│  biased, suggesting it may be                                                                                   │
│  fake.","agent_inputs_summary":{"claim_verdicts":["CONTRADICTED","SUPPORTED","UNVERIFIABLE"],"bias_score":0.7,  │
│  "tone":"sensationalist"}}                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Predicted: FAKE | confidence: 0.82


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 3. Next steps

Use `python scripts/run_batch.py --n 100` to process a 100-article sample and produce `results/results.csv`, then open `02_evaluation.ipynb` for metrics and error analysis.